# Import Libraries

In [2]:
import os
import numpy as np
import tensorflow as tf
import mlflow
import mlflow.keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from mlflow.tracking import MlflowClient
from tensorflow.keras.applications.resnet50 import preprocess_input
import mlflow, mlflow.keras



# ASL Dataset

In [20]:
import os

asl_path = r"C:\Users\User\.cache\kagglehub\datasets\grassknoted\asl-alphabet\versions\1"

asl_train_dir = os.path.join(asl_path, "asl_alphabet_train", "asl_alphabet_train")

print("✅ Dataset path:", asl_train_dir)
print("📁 Example folders inside:", os.listdir(asl_train_dir)[:10])


✅ Dataset path: C:\Users\User\.cache\kagglehub\datasets\grassknoted\asl-alphabet\versions\1\asl_alphabet_train\asl_alphabet_train
📁 Example folders inside: ['A', 'B', 'C', 'D', 'del', 'E', 'F', 'G', 'H', 'I']


# Datagens Setup

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input
import os

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

asl_path = r"C:\Users\User\.cache\kagglehub\datasets\grassknoted\asl-alphabet\versions\1"
asl_train_dir = os.path.join(asl_path, "asl_alphabet_train", "asl_alphabet_train")

# Train & val datagens
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.25,
    shear_range=0.15,
    brightness_range=[0.7, 1.3],
    channel_shift_range=40.0,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    asl_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=SEED
)

val_gen = val_datagen.flow_from_directory(
    asl_train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=SEED
)


Found 69600 images belonging to 29 classes.
Found 17400 images belonging to 29 classes.


# Load model

In [5]:
import mlflow.keras
import mlflow
MLFLOW_TRACKING_URI = None  
if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

MODEL_URI = "models:/ResNetASL/Production"   # registry URI
print("Loading model from:", MODEL_URI)
model = mlflow.keras.load_model(MODEL_URI)
model.summary()


Loading model from: models:/ResNetASL/Production


C:\Users\User\AppData\Roaming\Python\Python313\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
C:\Users\User\AppData\Roaming\Python\Python313\site-packages\mlflow\tracking\_model_registry\utils.py:215: FutureWarning: Filesystem model registry backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri)


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_3[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,647,133 (90.21 MB)

 Trainable params: 17,009,949 (64.89 MB)

 Non-trainable params: 6,637,184 (25.32 MB)

# Inspect Single batch accuracy

In [6]:
X_batch, y_batch = next(val_gen)   
preds = model.predict(X_batch)
pred_labels = preds.argmax(axis=1)
true_labels = y_batch.argmax(axis=1)

batch_accuracy = (pred_labels == true_labels).mean()
print("Single batch accuracy:", batch_accuracy)  


1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Single batch accuracy: 1.0


# predict across all validation batches

In [7]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

steps = int((val_gen.samples + BATCH_SIZE - 1) // BATCH_SIZE)
y_true_all = []
y_pred_all = []

val_gen.reset()  
for i in range(steps):
    Xb, yb = next(val_gen)
    preds = model.predict(Xb)
    y_pred_all.extend(preds.argmax(axis=1).tolist())
    y_true_all.extend(yb.argmax(axis=1).tolist())

y_true_all = np.array(y_true_all)
y_pred_all = np.array(y_pred_all)

overall_acc = (y_pred_all == y_true_all).mean()
print("Validation accuracy over full set:", overall_acc)

# Optional: per-class report
target_names = [k for k, v in sorted(val_gen.class_indices.items(), key=lambda x: x[1])]
print(classification_report(y_true_all, y_pred_all, target_names=target_names))

# Confusion matrix (large for multi-class)
cm = confusion_matrix(y_true_all, y_pred_all)
print("Confusion matrix shape:", cm.shape)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 806ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 796ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 804ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 814ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 841ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 819ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 842ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 835ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 827ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 826ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 819ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 842ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 782ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 793ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 780ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 844ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 785ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 807ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 792ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 801ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 807ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 800ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 781ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 805ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 762ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 

# Classification report

In [8]:
target_names = [k for k, v in sorted(val_gen.class_indices.items(), key=lambda x: x[1])]
print(classification_report(y_true_all, y_pred_all, target_names=target_names))


              precision    recall  f1-score   support

           A       1.00      1.00      1.00       600
           B       0.99      1.00      1.00       600
           C       1.00      1.00      1.00       600
           D       0.99      0.98      0.98       600
           E       0.97      0.99      0.98       600
           F       1.00      1.00      1.00       600
           G       0.99      0.97      0.98       600
           H       0.97      1.00      0.99       600
           I       0.96      0.92      0.94       600
           J       0.99      0.84      0.91       600
           K       1.00      0.95      0.97       600
           L       0.98      1.00      0.99       600
           M       0.99      0.98      0.99       600
           N       0.90      0.99      0.94       600
           O       0.94      0.97      0.95       600
           P       0.89      0.97      0.93       600
           Q       0.99      0.88      0.93       600
           R       0.69    

# confusion matrix

In [9]:
cm = confusion_matrix(y_true_all, y_pred_all)
print("Confusion matrix shape:", cm.shape)


Confusion matrix shape: (29, 29)


In [10]:
THRESHOLD = 0.90  

if overall_acc < THRESHOLD:
    print("⚠ ALERT: model drift detected!")
else:
    print("✓ Model healthy – no drift detected.")


✓ Model healthy – no drift detected.


In [11]:
# Put this BEFORE any mlflow.* call or mlflow import that triggers git_utils
import os

# Option A — SILENCE the Git import/check (quick & safe)
# This prevents the noisy "Failed to import Git" message
os.environ["GIT_PYTHON_REFRESH"] = "quiet"


# Optionally print to confirm
print("GIT_PYTHON_REFRESH =", os.environ.get("GIT_PYTHON_REFRESH"))
print("GIT_PYTHON_GIT_EXECUTABLE =", os.environ.get("GIT_PYTHON_GIT_EXECUTABLE"))


GIT_PYTHON_REFRESH = quiet
GIT_PYTHON_GIT_EXECUTABLE = None


# log evaluation metrics

In [12]:
mlflow.set_experiment("monitoring_experiments")
with mlflow.start_run():
    mlflow.log_param("model", "models:/ResNetASL/Production")
    mlflow.log_metric("validation_accuracy", float(overall_acc))
    np.save("confusion_matrix.npy", cm)
    mlflow.log_artifact("confusion_matrix.npy")
    print("Logged validation_accuracy to MLflow:", overall_acc)


Logged validation_accuracy to MLflow: 0.9427011494252874


# continuous monitoring

In [13]:
# --- MONITOR LOOP FIXED VERSION ---
def monitor_and_alert(val_gen, model, threshold=0.90, max_batches=3):
    import numpy as np
    import mlflow

    # --- collect sample batches properly ---
    X_list = []
    y_list = []

    val_gen.reset()

    for i in range(max_batches):
        try:
            Xb, yb = next(val_gen)
        except StopIteration:
            break

        X_list.append(Xb)      # Xb is shape (BATCH, 224,224,3)
        y_list.append(yb)

    # --- stack batches into correct shape ---
    X_eval = np.vstack(X_list)      # shape (N, 224, 224, 3)
    y_eval = np.vstack(y_list)      # shape (N, 29)

    # --- prediction ---
    preds = model.predict(X_eval, batch_size=32)
    acc = (preds.argmax(axis=1) == y_eval.argmax(axis=1)).mean()
    print("Monitoring accuracy:", acc)

    # --- MLflow logging ---
    mlflow.set_experiment("monitoring_experiments")
    with mlflow.start_run():
        mlflow.log_metric("monitor_accuracy", float(acc))

    # --- alert ---
    if acc < threshold:
        print(f"⚠ ALERT: accuracy {acc:.3f} < threshold {threshold}")


In [14]:
monitor_and_alert(val_gen, model, threshold=0.90)


3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 727ms/step
Monitoring accuracy: 1.0
